[Your Rate Limits depend on your Usage Tier.](https://platform.openai.com/settings/organization/limits)

In [1]:
import base64
import json
import logging
from pathlib import Path
from pprint import pformat

import pandas as pd
from openai import OpenAI

In [2]:
def setup_logging(INTERMEDIATE_PATH, PHASE):
    """
    Set up logging configuration.
    
    Args:
        INTERMEDIATE_PATH (str): Directory to store log files.
        PHASE (str): Phase of analysis ("prompt_opt", "final_pred", or "experimental").
        
    Returns:
        Configured logger instance.
    """
    log_dir = INTERMEDIATE_PATH / PHASE
    log_dir.mkdir(parents=True, exist_ok=True)
    
    log_file = log_dir / "3_1_openai.log"
    
    logging.basicConfig(
        level=logging.DEBUG,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

def get_response_schemas(OPENAI_RESPONSE_SCHEMAS_PATH, PROMPTING_TYPE):
    """
    Load and process response schemas from a JSON file based on the prompting type.

    Parameters:
        OPENAI_RESPONSE_SCHEMAS_PATH (str): Path to the JSON file containing response schemas.
        PROMPTING_TYPE (str): Type of prompting ('s_d', 'cot_d', 's_m', or 'cot_m').

    Returns:
        A dictionary containing the relevant response schema.
    """
    try:
        with open(OPENAI_RESPONSE_SCHEMAS_PATH, "r") as file:
            schemas = json.load(file)
    except FileNotFoundError:
        logger.error(f"File not found: {OPENAI_RESPONSE_SCHEMAS_PATH}")
        raise
    except json.JSONDecodeError:
        logger.error(f"Failed to decode JSON from file: {OPENAI_RESPONSE_SCHEMAS_PATH}")
        raise

    logger.info(f"Loaded response schemas from {OPENAI_RESPONSE_SCHEMAS_PATH}", )

    if PROMPTING_TYPE in ["s_d", "cot_d"]:
        response_schema_dict = {
            "response_schema_construction": schemas[PROMPTING_TYPE]["response_schema_construction"],
            "response_schema_currentuse": schemas[PROMPTING_TYPE]["response_schema_currentuse"],
            "response_schema_storeys": schemas[PROMPTING_TYPE]["response_schema_storeys"]
        }
        logger.debug(f"Response schema for {PROMPTING_TYPE} prompting:\n{pformat(response_schema_dict)}")
        logger.info("Selected response schema for prompting type: %s", PROMPTING_TYPE)

    elif PROMPTING_TYPE in ["s_m", "cot_m"]:
        response_schema_dict = {
            "response_schema_merged": schemas[PROMPTING_TYPE]["response_schema_merged"]
        }
        logger.debug(f"Response schema for {PROMPTING_TYPE} prompting:\n{pformat(response_schema_dict)}")
        logger.info("Selected merged response schema for prompting type: %s", PROMPTING_TYPE)

    else:
        logger.error(f"Unsupported prompting type: {PROMPTING_TYPE}")
        raise ValueError(f"Unsupported prompting type: {PROMPTING_TYPE}")

    return response_schema_dict

def typologies_prediction_helpers(TYPOLOGIES_PREDICTION_HELPER_PATH, PROMPTING_TYPE):
    """
    Load and process typologies prediction helpers from a JSON file based on the prompting type.

    Parameters:
        TYPOLOGIES_PREDICTION_HELPER_PATH (str): Path to the JSON file containing typologies prediction helpers.
        PROMPTING_TYPE (str): Type of prompting ('s_d', 'cot_d', 's_m', or 'cot_m').

    Returns:
        A tuple containing the token, prompt dictionary, system prompt dictionary, list of prompt keys, and dataset.
    """
    try:
        with open(TYPOLOGIES_PREDICTION_HELPER_PATH, "r") as file:
            data = json.load(file)[PROMPTING_TYPE]
    except FileNotFoundError:
        logger.error(f"File not found: {TYPOLOGIES_PREDICTION_HELPER_PATH}")
        raise
    except json.JSONDecodeError:
        logger.error(f"Failed to decode JSON from file: {TYPOLOGIES_PREDICTION_HELPER_PATH}")
        raise

    token = data["token"]
    logger.info(f"Token required with {PROMPTING_TYPE} prompting: {token}")

    if PROMPTING_TYPE in ["s_d", "cot_d"]:
        system_prompt_construction = base64.b64decode(data["system_prompt"]["construction"]).decode("utf-8")
        system_prompt_currentuse = base64.b64decode(data["system_prompt"]["currentuse"]).decode("utf-8")
        system_prompt_storeys = base64.b64decode(data["system_prompt"]["storeys"]).decode("utf-8")

        system_prompt_dict = {
            "construction": system_prompt_construction,
            "currentuse": system_prompt_currentuse,
            "storeys": system_prompt_storeys
        }

        logger.debug(f"Decoded system prompts for {PROMPTING_TYPE} prompting:\n{pformat(system_prompt_dict)}")

        prompt_construction = base64.b64decode(data["prompt"]["construction"]).decode("utf-8")
        prompt_currentuse = base64.b64decode(data["prompt"]["currentuse"]).decode("utf-8")
        prompt_storeys = base64.b64decode(data["prompt"]["storeys"]).decode("utf-8")

        prompt_dict = {
            "construction": prompt_construction,
            "currentuse": prompt_currentuse,
            "storeys": prompt_storeys
        }

        logger.debug(f"Decoded prompts for {PROMPTING_TYPE} prompting:\n{pformat(prompt_dict)}")

        prompt_dict_keys = list(prompt_dict.keys())

        dataset = pd.DataFrame({
            "id": [],
            "image_base64": [],
            "system_prompt_construction": [],
            "system_prompt_currentuse": [],
            "system_prompt_storeys": [],
            "prompt_construction": [],
            "prompt_currentuse": [],
            "prompt_storeys": []
        }).astype({
            "id": "int32",
            "image_base64": str,
            "system_prompt_construction": str,
            "system_prompt_currentuse": str,
            "system_prompt_storeys": str,
            "prompt_construction": str,
            "prompt_currentuse": str,
            "prompt_storeys": str
        })

    elif PROMPTING_TYPE in ["s_m", "cot_m"]:
        system_prompt_merged = base64.b64decode(data["system_prompt"]).decode("utf-8")

        system_prompt_dict = {
            "merged": system_prompt_merged
        }

        logger.debug(f"Decoded merged system prompt for {PROMPTING_TYPE} prompting:\n{pformat(system_prompt_dict)}")

        prompt_merged = base64.b64decode(data["prompt"]).decode("utf-8")

        prompt_dict = {
            "merged": prompt_merged
        }

        logger.debug(f"Decoded merged prompt for {PROMPTING_TYPE} prompting:\n{pformat(prompt_dict)}")

        prompt_dict_keys = list(prompt_dict.keys())

        dataset = pd.DataFrame({
            "id": [],
            "image_base64": [],
            "system_prompt_merged": [],
            "prompt_merged": []
        }).astype({
            "id": "int32",
            "image_base64": str,
            "system_prompt_merged": str,
            "prompt_merged": str
        })

    else:
        logger.error(f"Unsupported prompting type: {PROMPTING_TYPE}")
        raise ValueError(f"Unsupported prompting type: {PROMPTING_TYPE}")

    return token, prompt_dict, system_prompt_dict, prompt_dict_keys, dataset

def encode_image(image_path):
    """
    Encodes an image file into a base64 string.

    Args:
        image_path (str): Path to the image file.

    Returns:
        str: Base64 encoded string of the image.
    """
    try:
        with open(image_path, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
        logging.debug(f"Encoded image from {image_path}")
        return encoded_string
    except Exception as e:
        logging.error(f"Failed to encode image {image_path}: {e}")
        return None

def create_helper_df(ARD_GSV_PATH, PREDICT_COUNT, system_prompt_dict, prompt_dict, prompt_dict_keys, dataset):
    """
    Create a helper DataFrame by processing image files and updating the dataset with system prompts and prompts.

    Parameters:
        ARD_GSV_PATH (str): Path to the directory containing JPEG images.
        PREDICT_COUNT (int): Number of samples to predict.
        system_prompt_dict (dict): Dictionary containing system prompts.
        prompt_dict (dict): Dictionary containing prompts.
        prompt_dict_keys (list): List of keys for the prompt dictionaries (i.e., typologies: construction, currentuse, and storeys).
        dataset (DataFrame): Initial helper dataset to be updated.

    Returns:
        Updated dataset DataFrame with image IDs and associated prompts.
    """
    jpeg_images = list(Path(ARD_GSV_PATH).iterdir())
    if not jpeg_images:
        logger.warning(f"No JPEG images found in {ARD_GSV_PATH}")
        return dataset

    img_ids = []
    img_base64s = []
    
    for image in jpeg_images:
        if image.suffix == ".jpeg":
            img_id = int(image.name.split("_")[0])
            img_ids.append(img_id)

            img_base64 = encode_image(image)
            img_base64s.append(img_base64)
        else:
            logger.debug(f"Skipped non-JPEG file: {image.name}")

    dataset["id"] = img_ids
    dataset["image_base64"] = img_base64s
    logger.info(f"Loaded {len(img_ids)} image IDs from {ARD_GSV_PATH}")

    if not dataset["id"].is_unique:
        logger.error("Image IDs are not unique")
        raise ValueError("Duplicate image IDs found")

    if PROMPTING_TYPE in ["s_d", "cot_d"]:
        if len(prompt_dict_keys) != 3:
            logger.error("Prompt dictionary keys should be exactly three for 's_d' or 'cot_d'")
            raise ValueError("Invalid number of prompt dictionary keys")
        
        dataset["system_prompt_construction"] = system_prompt_dict[prompt_dict_keys[0]]
        dataset["system_prompt_currentuse"] = system_prompt_dict[prompt_dict_keys[1]]
        dataset["system_prompt_storeys"] = system_prompt_dict[prompt_dict_keys[2]]
        
        dataset["prompt_construction"] = prompt_dict[prompt_dict_keys[0]]
        dataset["prompt_currentuse"] = prompt_dict[prompt_dict_keys[1]]
        dataset["prompt_storeys"] = prompt_dict[prompt_dict_keys[2]]
        
        logger.info("Updated dataset with construction, current use, and storeys prompts")

    elif PROMPTING_TYPE in ["s_m", "cot_m"]:
        if len(prompt_dict_keys) != 1:
            logger.error("Prompt dictionary keys should be exactly one for 's_m' or 'cot_m'")
            raise ValueError("Invalid number of prompt dictionary keys")
        
        dataset["system_prompt_merged"] = system_prompt_dict[prompt_dict_keys[0]]
        
        dataset["prompt_merged"] = prompt_dict[prompt_dict_keys[0]]
        
        logger.info("Updated dataset with merged prompts")

    else:
        logger.error(f"Unsupported prompting type: {PROMPTING_TYPE}")
        raise ValueError(f"Unsupported prompting type: {PROMPTING_TYPE}")

    dataset.sort_values("id", inplace=True)
    dataset.reset_index(drop=True, inplace=True)

    if PREDICT_COUNT > len(dataset):
        logger.warning(f"PREDICT_COUNT ({PREDICT_COUNT}) is greater than the number of available images ({len(dataset)}). Using all available images.")
        PREDICT_COUNT = len(dataset)

    dataset = dataset.sample(PREDICT_COUNT, random_state=123)
    logger.info(f"Sampled {PREDICT_COUNT} rows from the dataset")

    return dataset

def get_task_partitions(NUM_PARTITION, response_schema_dict, prompt_dict_keys, dataset, token):
    """
    Partition tasks based on the number of partitions and generate task dictionaries.
    
    Parameters:
        NUM_PARTITION (int): Number of partitions.
        response_schema_dict (dict): Dictionary containing relevant response schemas.
        prompt_dict_keys (list): List of keys for prompts.
        dataset (pd.DataFrame): DataFrame containing the dataset.
        token (str): Token required for prompting.
    
    Returns:
        dict: Dictionary containing task partitions.
    """
    # Add custom_id column
    dataset['custom_id'] = 'task-' + dataset.id.astype(str)    
    logger.info("Added custom_id column to the dataset")

    # Generate messages for each prompt type
    for typology in prompt_dict_keys:
        dataset[f'messages_{typology}'] = dataset.apply(
            lambda row: [
                {
                    "role": "system",
                    "content": row[f'system_prompt_{typology}']
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{row['image_base64']}"
                            }
                        },
                        {
                            "type": "text",
                            "text": row[f'prompt_{typology}']
                        }
                    ],
                }
            ], axis=1
        )
        logger.debug(f"Generated messages for {typology} prompt type")
    
    for typology in prompt_dict_keys:
        dataset[f'task_{typology}'] = dataset.apply(
            lambda row: {
                "custom_id": row['custom_id'],
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "gpt-4o-2024-11-20",
                    "max_completion_tokens": token + 65,  # 65 accounting estimated tokens required to output the JSON's keys, brackets, commas, etc.,
                    "seed": 123,
                    "response_format": response_schema_dict[f"response_schema_{typology}"],
                    "messages": row[f'messages_{typology}']
                }
            }, axis=1
        )
        logger.debug(f"Generated tasks for {typology} prompt type")
    
    tasks_dict = {}

    # Partition tasks
    for typology in prompt_dict_keys:
        task_list = dataset[f"task_{typology}"].tolist()
        task_partition_len = len(task_list) // NUM_PARTITION
        for i in range(0, NUM_PARTITION):
            task_list_start = task_partition_len * i
            task_list_end = task_partition_len * (i + 1) if i != NUM_PARTITION - 1 else len(task_list)
            tasks_dict[f"{typology}_partition_{i + 1}"] = task_list[task_list_start:task_list_end]
        logger.debug(f"Partitioned tasks for {typology} prompt type into {NUM_PARTITION} partitions")
    
    return tasks_dict

def submit_batch_requests(NUM_PARTITION, PROMPTING_TYPE, prompt_dict_keys, tasks_dict):
    """
    Submit batch requests for predictions and store the response IDs.

    Parameters:
        NUM_PARTITION (int): Number of partitions to divide tasks into.
        PROMPTING_TYPE (str): Type of prompting.
        prompt_dict_keys (list): List of keys for the prompt dictionaries (i.e., typologies: construction, currentuse, and storeys).
        tasks_dict (dict): Dictionary containing task information.

    Returns:
        A dictionary with batch response IDs.
    """
    batch_file_dict = {}
    batch_job_dict = {}
    
    for typology in prompt_dict_keys:
        for i in range(NUM_PARTITION):
            file_name = INTERMEDIATE_DIR / PHASE / f"o_{PROMPTING_TYPE}_{typology}_partition_{i}.jsonl"

            # Write tasks to file
            with open(file_name, 'w') as file:
                for obj in tasks_dict[f"{typology}_partition_{i + 1}"]:
                    file.write(json.dumps(obj) + '\n')

            # Upload file to client storage
            try:
                batch_file_dict[f"o_{PROMPTING_TYPE}_{typology}_partition_{i}"] = client.files.create(
                    file=open(file_name, "rb"),
                    purpose="batch"
                )
                logging.debug(f"File {file_name} uploaded successfully.")
            except Exception as e:
                logging.error(f"Failed to upload file {file_name}: {e}")
                continue
        
            # Create batch job
            try:
                batch_job_dict[f"o_{PROMPTING_TYPE}_{typology}_partition_{i}"] = client.batches.create(
                    input_file_id=batch_file_dict[f"o_{PROMPTING_TYPE}_{typology}_partition_{i}"].id,
                    endpoint="/v1/chat/completions",
                    completion_window="24h"
                )
                logging.debug(f"Batch job created for partition {i} of typology {typology}.")
            except Exception as e:
                logging.error(f"Failed to create batch job for partition {i} of typology {typology}: {e}")
                continue
    
    return batch_job_dict

def monitor_batch(batch_id, partition, typology):
    """
    Monitor the status of a batch request.

    Parameters:
        batch_id (str): The ID of the batch to monitor.
        partition (int): The partition number.
        typology (str): Building typology predicted.

    Returns:
        The status of the batch.
    """
    try:
        batch_job = client.batches.retrieve(batch_id)
        logging.debug(f"Status of batch job for {typology} #{partition} partition: {batch_job.status}")
        return batch_job.status
    except Exception as e:
        logging.error(f"Failed to retrieve batch job status for {typology} #{partition} partition: {e}")
        return None

def process_results(NUM_PARTITION, batch_id, batch_metadata_dict):
    """
    Processes the results of a message batch and returns them as a DataFrame.

    Args:
        NUM_PARTITION (int): Number of partitions to divide tasks into.
        batch_id (str): ID of the message batch.
        batch_metadata_dict (dict): Metadata dictionary for the batch.

    Returns:
        A DataFrame containing processed results.
    """
    df_list = []
    for typology in prompt_dict_keys:
        for i in range(NUM_PARTITION):
            try:
                batch_job = client.batches.retrieve(batch_job_dict[f"o_{PROMPTING_TYPE}_{typology}_partition_{i}"].id)
                result_file_id = batch_job.output_file_id
                result = client.files.content(result_file_id).content
                
                result_file_name = INTERMEDIATE_DIR / PHASE / f"result_o_{PROMPTING_TYPE}_{typology}_partition_{i}.jsonl"
                
                with open(result_file_name, 'wb') as file:
                    file.write(result)
                
                # Loading data from saved file
                results = []
                with open(result_file_name, 'r') as file:
                    for line in file:
                        # Parsing the JSON string into a dict and appending to the list of results
                        json_object = json.loads(line.strip())
                        results.append(json_object)
                
                # Reading only the first results
                id_list = []
                json_list = []
                
                for res in results:
                    res_id = int(res['custom_id'].split("-")[1])
                    res_json = json.loads(res['response']['body']['choices'][0]['message']['content'])
        
                    id_list.append(res_id)
                    json_list.append(res_json)
                    
                df = pd.DataFrame(json_list)
                df["id"] = id_list
        
                df_list.append(df)
                logging.debug(f"Processed results for partition {i} of typology {typology}.")

            except Exception as e:
                logging.error(f"Failed to process results for partition {i} of typology {typology}: {e}")
                continue
    
    return df_list

def merge_and_save_predictions(INTERMEDIATE_DIR, OUTPUT_DIR, PROMPTING_TYPE, PHASE, df_list, prompt_dict_keys):
    """
    Merges prediction DataFrames and saves them to an Excel file.

    Args:
        INTERMEDIATE_DIR (Path): Directory containing the JSONL files.
        OUTPUT_DIR (Path): Directory where the output Excel file will be saved.
        PROMPTING_TYPE (str): Type of prompting used.
        PHASE (str): Phase of analysis ("prompt_opt", "final_pred", or "experimental").
        df_list (list): A list of DataFrames containing processed results.
        prompt_dict_keys (list): List of keys for the prompt dictionaries (i.e., typologies: construction, currentuse, and storeys).

    Returns:
        Prediction results saved.
    """
    df_master = pd.concat(df_list)
    df_list = []
    
    if PROMPTING_TYPE in ["s_d", "s_m"]:
        rename_dict = {
            "reason_x": "construction_reason",
            "reason_y": "currentuse_reason",
            "reason": "storeys_reason"
        }
    
    elif PROMPTING_TYPE in ["cot_d", "cot_m"]:
        rename_dict = {
            "reason_x": "construction_reason",
            "reason_y": "currentuse_reason",
            "thoughts_x": "construction_thoughts",
            "thoughts_y": "currentuse_thoughts",
            "thoughts": "storey_thoughts",
            "reason": "storeys_reason",
        }

    try:
        if PROMPTING_TYPE in ["s_d", "cot_d"]:
            for typology in prompt_dict_keys:
                if PROMPTING_TYPE in ["s_d"]:
                    prompt_type_df = df_master[~df_master[typology].isnull()].reset_index(drop=True)[["id", typology, "reason"]]
        
                if PROMPTING_TYPE in ["cot_d"]:
                    prompt_type_df = df_master[~df_master[typology].isnull()].reset_index(drop=True)[["id", f"thoughts_{typology}", typology, f"reason_{typology}"]]
                
                df_list.append(prompt_type_df)
        
        elif PROMPTING_TYPE in ["s_m", "cot_m"]:
            typologies = ["construction", "currentuse", "storeys"]
            for typology in typologies:
                if PROMPTING_TYPE in ["s_m"]:
                    prompt_type_df = df_master[~df_master[typology].isnull()].reset_index(drop=True)[["id", typology, "reason"]]
        
                if PROMPTING_TYPE in ["cot_m"]:
                    prompt_type_df = df_master[~df_master[typology].isnull()].reset_index(drop=True)[["id", f"thoughts_{typology}", typology, f"reason_{typology}"]]
                
                df_list.append(prompt_type_df)
                
        merged_df = df_list[0]
        for df in df_list[1:-1]:
            merged_df = pd.merge(merged_df, df, on='id', how='outer')
        
        merged_df.rename(rename_dict, axis=1, inplace=True)
        
        out_path = OUTPUT_DIR / PHASE / f"{PROMPTING_TYPE}_openai.xlsx"
        merged_df[~merged_df.id.duplicated()].to_excel(out_path, index=False)
        logging.info(f"Merged predictions saved to {out_path}")

    except Exception as e:
        logging.error(f"Failed to merge and save predictions: {e}")

In [3]:
# Configuration
INTERMEDIATE_DIR = Path("./temp/3_1")
OUTPUT_DIR = Path("./output/3_1")

# Create directories if they don't exist
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPENAI_RESPONSE_SCHEMAS_PATH = "./helper/openai_response_schemas.json"
TYPOLOGIES_PREDICTION_HELPER_PATH = "./helper/typologies_prediction_helper.json"
ARD_GSV_PATH = "./output/2/ard"

# Initializing OpenAI client - see https://platform.openai.com/docs/quickstart?context=python
with open('./helper/openai_key.json') as j:
    openai_token = json.load(j)

API_KEY = openai_token['api_key']
CLIENT = OpenAI(api_key=API_KEY)

In [4]:
PROMPTING_TYPE = "s_m"  # s_d: straight divided, cot_d: CoT divided, s_m: straight merged, cot_m: CoT merged

PREDICT_COUNT = 5  # GSV image to predict
if PREDICT_COUNT == 2000:
    PHASE = "prompt_opt"
elif PREDICT_COUNT == 30000:
    PHASE = "final_pred"
else:
    PHASE = "experimental"

logger = setup_logging(INTERMEDIATE_DIR, PHASE)

In [5]:
NUM_PARTITION = 2  # Adjust later according to your needs and rate limit

In [6]:
response_schema_dict = get_response_schemas(OPENAI_RESPONSE_SCHEMAS_PATH, PROMPTING_TYPE)
token, prompt_dict, system_prompt_dict, prompt_dict_keys, base_dataset = typologies_prediction_helpers(TYPOLOGIES_PREDICTION_HELPER_PATH, PROMPTING_TYPE)

2025-10-27 15:22:22,547 - INFO - Loaded response schemas from ./helper/openai_response_schemas.json
2025-10-27 15:22:22,551 - DEBUG - Response schema for s_m prompting:
{'response_schema_merged': {'json_schema': {'name': 'merged_response',
                                            'schema': {'additionalProperties': False,
                                                       'properties': {'construction': {'type': 'string'},
                                                                      'currentuse': {'type': 'string'},
                                                                      'reason_construction': {'type': 'string'},
                                                                      'reason_currentuse': {'type': 'string'},
                                                                      'reason_storeys': {'type': 'string'},
                                                                      'storeys': {'type': 'string'}},
                              

In [ ]:
dataset = create_helper_df(ARD_GSV_PATH, PREDICT_COUNT, system_prompt_dict, prompt_dict, prompt_dict_keys, base_dataset)
tasks_dict = get_task_partitions(NUM_PARTITION, response_schema_dict, prompt_dict_keys, dataset, token)

In [ ]:
submit_batch_requests(NUM_PARTITION, PROMPTING_TYPE, prompt_dict_keys, tasks_dict)

In [ ]:
monitor_batch(batch_id, partition, typology)  # You can re-run this code until every requests are finished

In [ ]:
process_results(NUM_PARTITION, batch_id, batch_metadata_dict)
merge_and_save_predictions(INTERMEDIATE_DIR, OUTPUT_DIR, PROMPTING_TYPE, PHASE, df_list, prompt_dict_keys)